# 07 — Visualisasi Hasil (Bab 2.4c)Notebook ini membuat visualisasi dari hasil analisis menggunakan **Pandas + Matplotlib + Seaborn**.

## 7.1 Inisialisasi & Load Data

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("07_Visualization") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

# Load data
df = spark.read.parquet("/output/retail_parquet")
df.createOrReplaceTempView("transactions")

df_segments = spark.read.parquet("/output/customer_segments")

print(f"✅ Transactions: {df.count()} rows")
print(f"✅ Segments    : {df_segments.count()} rows")

## 7.2 Setup Matplotlib

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import pandas as pd

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11

print("✅ Matplotlib & Seaborn ready")

## 7.3 Chart 1: Revenue per Kategori Produk

In [ ]:
pdf_revenue = spark.sql("""
    SELECT Product_Category, SUM(Total_Amount) as revenue, COUNT(*) as count
    FROM transactions GROUP BY Product_Category ORDER BY revenue DESC
""").toPandas()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#4ECDC4", "#45B7D1", "#FF6B6B"]
bars = ax.bar(pdf_revenue["Product_Category"], pdf_revenue["revenue"], color=colors, edgecolor="white", linewidth=1.5)

# Tambah label di atas bar
for bar, rev, cnt in zip(bars, pdf_revenue["revenue"], pdf_revenue["count"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
            f'{rev:,.0f}\n({cnt} txn)', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_title("Total Revenue per Kategori Produk", fontsize=14, fontweight='bold', pad=20)
ax.set_ylabel("Revenue", fontsize=12)
ax.ticklabel_format(style='plain', axis='y')
ax.set_ylim(0, max(pdf_revenue["revenue"]) * 1.15)
sns.despine()
plt.tight_layout()
plt.savefig("/home/jovyan/work/chart_revenue_kategori.png", dpi=150)
plt.show()
print("✅ Saved: chart_revenue_kategori.png")

## 7.4 Chart 2: Tren Penjualan Bulanan

In [ ]:
pdf_monthly = spark.sql("""
    SELECT MONTH(Date) as bulan, SUM(Total_Amount) as revenue, COUNT(*) as transaksi
    FROM transactions WHERE YEAR(Date) = 2023
    GROUP BY MONTH(Date) ORDER BY bulan
""").toPandas()

fig, ax1 = plt.subplots(figsize=(10, 5))

# Revenue line
color1 = "#6C5CE7"
ax1.plot(pdf_monthly["bulan"], pdf_monthly["revenue"], marker='o', color=color1, linewidth=2.5, markersize=8, label="Revenue")
ax1.fill_between(pdf_monthly["bulan"], pdf_monthly["revenue"], alpha=0.1, color=color1)
ax1.set_xlabel("Bulan", fontsize=12)
ax1.set_ylabel("Revenue", fontsize=12, color=color1)
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(["Jan","Feb","Mar","Apr","Mei","Jun","Jul","Agu","Sep","Okt","Nov","Des"])

# Transaction count on secondary axis
ax2 = ax1.twinx()
color2 = "#FFA726"
ax2.bar(pdf_monthly["bulan"], pdf_monthly["transaksi"], alpha=0.3, color=color2, label="Jumlah Transaksi")
ax2.set_ylabel("Jumlah Transaksi", fontsize=12, color=color2)

ax1.set_title("Tren Penjualan Bulanan (2023)", fontsize=14, fontweight='bold', pad=15)
fig.legend(loc="upper right", bbox_to_anchor=(0.95, 0.95))
plt.tight_layout()
plt.savefig("/home/jovyan/work/chart_tren_bulanan.png", dpi=150)
plt.show()
print("✅ Saved: chart_tren_bulanan.png")

## 7.5 Chart 3: Distribusi Usia Pelanggan

In [ ]:
pdf_age = spark.sql("SELECT Age FROM transactions").toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(pdf_age["Age"], bins=20, color="#4ECDC4", edgecolor="white", linewidth=1.2, alpha=0.8)
ax.axvline(pdf_age["Age"].mean(), color="#FF6B6B", linestyle="--", linewidth=2, label=f'Mean: {pdf_age["Age"].mean():.1f}')
ax.set_title("Distribusi Usia Pelanggan", fontsize=14, fontweight='bold')
ax.set_xlabel("Usia", fontsize=12)
ax.set_ylabel("Frekuensi", fontsize=12)
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.savefig("/home/jovyan/work/chart_distribusi_usia.png", dpi=150)
plt.show()
print("✅ Saved: chart_distribusi_usia.png")

## 7.6 Chart 4: Segmentasi Pelanggan (K-Means Scatter)

In [ ]:
pdf_seg = df_segments.toPandas()

fig, ax = plt.subplots(figsize=(9, 6))
colors_seg = {0: "#FF6B6B", 1: "#4ECDC4", 2: "#45B7D1"}
labels_seg = {0: "Segment 0", 1: "Segment 1", 2: "Segment 2"}

for seg in sorted(pdf_seg["segment"].unique()):
    mask = pdf_seg["segment"] == seg
    ax.scatter(
        pdf_seg[mask]["frequency"], 
        pdf_seg[mask]["avg_monetary"],
        c=colors_seg.get(seg, "#999"),
        label=labels_seg.get(seg, f"Seg {seg}"),
        alpha=0.6, s=60, edgecolors="white", linewidth=0.5
    )

ax.set_title("Segmentasi Pelanggan (K-Means, k=3)", fontsize=14, fontweight='bold')
ax.set_xlabel("Frequency (Jumlah Transaksi)", fontsize=12)
ax.set_ylabel("Average Monetary (Avg Spending)", fontsize=12)
ax.legend(fontsize=11, title="Segment")
sns.despine()
plt.tight_layout()
plt.savefig("/home/jovyan/work/chart_segmentasi_kmeans.png", dpi=150)
plt.show()
print("✅ Saved: chart_segmentasi_kmeans.png")

## 7.7 Chart 5: Heatmap Gender × Kategori

In [ ]:
pdf_cross = spark.sql("""
    SELECT Gender, Product_Category, ROUND(AVG(Total_Amount), 1) as avg_spending
    FROM transactions
    GROUP BY Gender, Product_Category
""").toPandas()

pivot = pdf_cross.pivot(index="Gender", columns="Product_Category", values="avg_spending")

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=2, linecolor="white",
            annot_kws={"size": 14, "weight": "bold"}, ax=ax)
ax.set_title("Avg Spending: Gender × Kategori Produk", fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig("/home/jovyan/work/chart_heatmap_gender.png", dpi=150)
plt.show()
print("✅ Saved: chart_heatmap_gender.png")

## 7.8 Chart 6: 3D Scatter Segmentasi Pelanggan

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for seg in sorted(pdf_seg["segment"].unique()):
    mask = pdf_seg["segment"] == seg
    ax.scatter(pdf_seg[mask]["frequency"], 
               pdf_seg[mask]["avg_monetary"],
               pdf_seg[mask]["avg_quantity"],
               c=colors_seg.get(seg, "#999"), label=f"Seg {seg}",
               alpha=0.6, s=40, edgecolors="white", linewidth=0.5)

ax.set_title("3D Segmentasi Pelanggan (K-Means)", fontsize=14, fontweight='bold')
ax.set_xlabel("Frequency", labelpad=10)
ax.set_ylabel("Avg Monetary", labelpad=10)
ax.set_zlabel("Avg Quantity", labelpad=10)
ax.legend(fontsize=11, title="Segment", loc="upper left")
plt.tight_layout()
plt.savefig("/home/jovyan/work/chart_segmentasi_3d.png", dpi=150)
plt.show()
print("✅ Saved: chart_segmentasi_3d.png")

## 7.9 Dashboard Summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Dashboard Analisis Penjualan Ritel", fontsize=18, fontweight='bold', y=1.02)

# 1. Revenue per kategori
colors = ["#4ECDC4", "#45B7D1", "#FF6B6B"]
axes[0,0].bar(pdf_revenue["Product_Category"], pdf_revenue["revenue"], color=colors)
axes[0,0].set_title("Revenue per Kategori", fontweight='bold')
axes[0,0].ticklabel_format(style='plain', axis='y')

# 2. Tren bulanan
axes[0,1].plot(pdf_monthly["bulan"], pdf_monthly["revenue"], marker='o', color="#6C5CE7", linewidth=2)
axes[0,1].set_title("Tren Bulanan", fontweight='bold')
axes[0,1].set_xticks(range(1, 13))

# 3. Distribusi usia
axes[1,0].hist(pdf_age["Age"], bins=20, color="#4ECDC4", edgecolor="white")
axes[1,0].set_title("Distribusi Usia", fontweight='bold')

# 4. Segmentasi
for seg in sorted(pdf_seg["segment"].unique()):
    mask = pdf_seg["segment"] == seg
    axes[1,1].scatter(pdf_seg[mask]["frequency"], pdf_seg[mask]["avg_monetary"],
                      c=colors_seg.get(seg), label=f"Seg {seg}", alpha=0.6, s=40)
axes[1,1].set_title("Segmentasi Pelanggan", fontweight='bold')
axes[1,1].legend()

for ax in axes.flat:
    sns.despine(ax=ax)

plt.tight_layout()
plt.savefig("/home/jovyan/work/dashboard_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard saved: dashboard_summary.png")

## 7.10 Cleanup

In [ ]:
spark.stop()
print("✅ Semua visualisasi selesai.")
print("📂 File tersimpan di folder notebooks/:")
print("   - chart_revenue_kategori.png")
print("   - chart_tren_bulanan.png")
print("   - chart_distribusi_usia.png")
print("   - chart_segmentasi_kmeans.png")
print("   - chart_heatmap_gender.png")
print("   - chart_segmentasi_3d.png")
print("   - dashboard_summary.png")